In [ ]:
import itertools
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import seaborn as sns
from pyproj import datadir

# datadir.set_data_dir("/usr/share/proj")

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Outlook": [
        "Sunny", "Sunny", "Overcast", "Rainy", "Rainy", "Rainy",
        "Overcast", "Sunny", "Sunny", "Rainy", "Sunny",
        "Overcast", "Overcast", "Rainy"
    ],
    "Temperature": [
        "hot", "hot", "hot", "mild", "cool", "cool",
        "cool", "mild", "cool", "mild", "mild",
        "mild", "hot", "mild"
    ],
    "Humidity": [
        "high", "high", "high", "high", "normal", "normal",
        "normal", "high", "normal", "normal", "normal",
        "high", "normal", "high"
    ],
    "Windy": [
        False, True, False, False, False, True,
        True, False, False, False, True,
        True, False, True
    ],
    "Play": [
        "no", "no", "yes", "yes", "yes", "no",
        "yes", "no", "yes", "yes", "yes",
        "yes", "yes", "no"
    ]
})

df

In [ ]:
features = ["Outlook", "Temperature", "Humidity", "Windy"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, feature in zip(axes.ravel(), features):

    sns.countplot(
        data=df,
        x=feature,
        ax=ax
    )

    ax.set_title(feature)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
features = ["Outlook", "Temperature", "Humidity", "Windy"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, feature in zip(axes.ravel(), features):

    sns.countplot(
        data=df,
        x=feature,
        hue="Play",
        ax=ax
    )

    ax.set_title(feature)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

# Clouds

In [ ]:
# https://www.scielo.br/j/bcg/a/jGFBvBWTVz6L56Yb73DQKFr/?format=html&lang=pt

# https://data.inpe.br/stac/browser/collections/CB4A-MUX-L4-DN-1/items/CBERS_4A_MUX_20260108_202_142_L4?.language=en
# https://data.inpe.br/stac/browser/collections/CB4A-MUX-L4-SR-1/items/CBERS_4A_MUX_20260108_202_142_L4?.language=en

In [ ]:
gpkg = "/home/jovyan/Disciplinas/Data_Science/09_dt/samples.gpkg"

bands = {
    "blue": "/home/jovyan/Disciplinas/Data_Science/09_dt/imgs/CBERS_4A_MUX_20260108_202_142_L4_BAND5_GRID_SURFACE.tif",
    "green": "/home/jovyan/Disciplinas/Data_Science/09_dt/imgs/CBERS_4A_MUX_20260108_202_142_L4_BAND6_GRID_SURFACE.tif",
    "red": "/home/jovyan/Disciplinas/Data_Science/09_dt/imgs/CBERS_4A_MUX_20260108_202_142_L4_BAND7_GRID_SURFACE.tif",
    "nir": "/home/jovyan/Disciplinas/Data_Science/09_dt/imgs/CBERS_4A_MUX_20260108_202_142_L4_BAND8_GRID_SURFACE.tif",
}

# bands = {
#     "blue": "/home/jovyan/Disciplinas/Data_Science/09_dt/imgs/CBERS_4A_MUX_20260108_202_142_L4_BAND5.tif",
#     "green": "/home/jovyan/Disciplinas/Data_Science/09_dt/imgs/CBERS_4A_MUX_20260108_202_142_L4_BAND6.tif",
#     "red": "/home/jovyan/Disciplinas/Data_Science/09_dt/imgs/CBERS_4A_MUX_20260108_202_142_L4_BAND7.tif",
#     "nir": "/home/jovyan/Disciplinas/Data_Science/09_dt/imgs/CBERS_4A_MUX_20260108_202_142_L4_BAND8.tif",
# }

classes = "class"

In [ ]:
gdf = gpd.read_file(gpkg)

if gdf.crs is None:
    raise ValueError("no CRS defined.")

# Keep original lon lat
gdf["lon"] = gdf.geometry.x
gdf["lat"] = gdf.geometry.y

with rasterio.open(next(iter(bands.values()))) as src:
    raster_crs = src.crs

if gdf.crs != raster_crs:
    gdf = gdf.to_crs(raster_crs)

coords = [(geom.x, geom.y) for geom in gdf.geometry]

for name, file in bands.items():
    with rasterio.open(file) as src:
        valores = []
        for valor in src.sample(coords):
            v = valor[0]
            if src.nodata is not None and v == src.nodata:
                valores.append(None)
            else:
                valores.append(v)
        gdf[name] = valores

colunas = [
    "lon",
    "lat",
    classes,
    "blue",
    "green",
    "red",
    "nir",
]

df = pd.DataFrame(gdf[colunas])

df

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, band in zip(axes.ravel(), bands):

    sns.histplot(
        data=df,
        x=band,
        bins=50,
        edgecolor="black",
        ax=ax
    )

    ax.set_title(f"{band.capitalize()} Band")
    ax.set_xlabel("Pixel Value")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
palette = {
    "cloud": "white",
    "shadow": "black",
    "clear": "yellow"
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, band in zip(axes.ravel(), bands):

    sns.histplot(
        data=df,
        x=band,
        hue="class",
        palette=palette,
        bins=50,
        stat="count",
        common_norm=False,
        alpha=0.6,
        edgecolor="black",
        ax=ax
    )

    ax.set_title(f"{band.capitalize()} Band")
    ax.set_xlabel("Reflectance")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

sns.scatterplot( #regplot
    data=df,
    x='red',
    y='nir',
    # scatter_kws={'alpha':0.7},
    # line_kws={'color':'red'}
)

plt.title('Scatterplot')
plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(
    data=df,
    x="red",
    y="nir",
    hue="class",
    palette=palette,
    edgecolor="gray",
    linewidth=0.3,
    s=40
)

plt.title("Red vs NIR")
plt.legend(title="Class")
plt.tight_layout()
plt.show()

In [ ]:
pairs = list(itertools.combinations(bands, 2))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (x_band, y_band) in zip(axes.ravel(), pairs):
    sns.scatterplot(
        data=df,
        x=x_band,
        y=y_band,
        hue="class",
        palette=palette,
        edgecolor="gray",
        linewidth=0.3,
        s=25,
        legend=False,
        ax=ax
    )

    ax.set_title(f"{x_band.capitalize()} vs {y_band.capitalize()}")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title="Class", loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
import random

num = random.randint(1, 10)
num

In [ ]:
random.seed(42)

num = random.randint(1, 10)
num

In [ ]:
df

# Data partition

In [ ]:
train = df[:41]
test = df[41:]

In [ ]:
train

In [ ]:
test

In [ ]:
x = df.drop(columns='class')

y = df['class']

x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

print(f"Amostras de treino: {len(x_train)}")
print(f"Amostras de validação: {len(x_val)}")

In [ ]:
x_train

In [ ]:
x_val

In [ ]:
y_train

In [ ]:
y_val

In [ ]:
print(y_train.value_counts())
print(y_val.value_counts())

In [ ]:
x = df.drop(columns='class')

y = df['class']

x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"training samples: {len(x_train)}")
print(f"validation samples: {len(x_val)}")

In [ ]:
x_train

In [ ]:
x_val

In [ ]:
y_train

In [ ]:
y_val

In [ ]:
print(y_train.value_counts())
print(y_val.value_counts())

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

In [ ]:
skf = StratifiedShuffleSplit(n_splits=2, train_size=0.8, random_state=42)

for i, (train_idx, val_idx) in enumerate(skf.split(df, df["class"]), start=1):
    print(f"Split {i}")
    print(f"N training samples: {len(train_idx)}")
    print(f"N validation samples: {len(val_idx)}")

    print("\training class distribution:")
    print(df.iloc[train_idx]["class"].value_counts())

    print("\nvalidation class distribution:")
    print(df.iloc[val_idx]["class"].value_counts())

    print()

In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import accuracy_score

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = RandomForestClassifier(
    random_state=42
)

scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(x), start=1):

    x_train = x.iloc[train_idx]
    x_val = x.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model.fit(x_train, y_train)

    y_pred = model.predict(x_val)

    acc = accuracy_score(y_val, y_pred)

    scores.append(acc)

    print(f"Fold {fold}")
    print(f"Acurácia: {acc:.3f}")
    print()

print(f"Average: {np.mean(scores):.3f}")
print(f"Std: {np.std(scores):.3f}")

In [ ]:
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(random_state=42)

scores = cross_val_score(
    rf,
    x,
    y,
    cv=5,
    scoring="accuracy"
)

print(scores)
print("Mean:", scores.mean())

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rf,
    x,
    y,
    cv=cv,
    scoring="accuracy"
)

print(scores)
print("Mean:", scores.mean())

In [ ]:
rf.get_params()

In [ ]:
rf.fit(x, y)

print(rf.estimators_)

In [ ]:
rf.fit(x, y)

print(rf.estimators_)

In [ ]:
tree1 = rf.estimators_[0]

print(tree1)

In [ ]:
tree1.get_depth()

In [ ]:
tree1.get_n_leaves()

In [ ]:
from sklearn.tree import export_text

print(
    export_text(
        tree1,
        feature_names=list(x.columns)
    )
)

In [ ]:
rf.estimators_samples_[0]

In [ ]:
x = df.drop(columns=['class', "lat", "lon"])
x

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rf,
    x,
    y,
    cv=cv,
    scoring="accuracy"
)

print(scores)
print("Mean:", scores.mean())

In [ ]:
x

In [ ]:
rf.fit(x, y)

tree1 = rf.estimators_[0]

print(
    export_text(
        tree1,
        feature_names=list(x.columns)
    )
)

scores = cross_val_score(
    rf,
    x,
    y,
    cv=cv,
    scoring="accuracy"
)

print(scores)
print("Mean:", scores.mean())

# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    criterion="gini",
    max_depth=None,
    random_state=42
)

cv = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    dt,
    x,
    y,
    cv=cv,
    scoring="accuracy"
)

print(scores)
print("Mean:", scores.mean())

In [ ]:
dt.fit(x, y)
print(dt.get_depth())

In [ ]:
print(dt.get_n_leaves())

In [ ]:
for feature in dt.tree_.feature:
    if feature >= 0:
        print(x.columns[feature])

In [ ]:
rules = export_text(
    dt,
    feature_names=list(x.columns)
)

print(rules)

In [ ]:
importance = pd.DataFrame({
    "band": x.columns,
    "importance": dt.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
)

In [ ]:
df["ndvi"] = (df["nir"] - df["red"]) / (df["nir"] + df["red"])
df["M"] = (
    0.25 * df["blue"] +
    0.375 * df["green"] +
    0.375 * df["red"]
)
df["wi"] = (
    abs(df["blue"] - df["M"]) / df["M"] +
    abs(df["green"] - df["M"]) / df["M"] +
    abs(df["red"] - df["M"]) / df["M"]
)
df["hot"] = (
    df["blue"] -
    0.45 * df["red"] -
    0.08
)

# df["d"] = df["B2"] < x and df["B4"] < y
# df["w"] = df["ndvi"] < ndvi_clean and df["B4"] < rhoclean
# df["diff"] = (df["NIR"] - df["NIR"])

x = df.drop(columns=['class', "lat", "lon", "M"])
x

In [ ]:
dt = DecisionTreeClassifier(
    criterion="gini",
    max_depth=None,
    random_state=42
)

cv = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    dt,
    x,
    y,
    cv=cv,
    scoring="accuracy"
)

print(scores)
print("Mean:", scores.mean())

dt.fit(x, y)
print(dt.get_depth())

print(dt.get_n_leaves())

for feature in dt.tree_.feature:
    if feature >= 0:
        print(x.columns[feature])

rules = export_text(
    dt,
    feature_names=list(x.columns)
)

print(rules)

In [ ]:
bands["blue"]

In [ ]:
blue = rasterio.open(bands["blue"]).read(1)
green = rasterio.open(bands["green"]).read(1)
red = rasterio.open(bands["red"]).read(1)
nir = rasterio.open(bands["nir"]).read(1)

In [ ]:
df["class"]

In [ ]:
y

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df["class"])

dt = DecisionTreeClassifier(
    criterion="gini",
    random_state=42
)

dt.fit(x, y)

In [ ]:
ndvi = (nir - red) / (nir + red)

M = (
    0.25 * blue +
    0.375 * green +
    0.375 * red
)

wi = (
    np.abs(blue - M) / (M) +
    np.abs(green - M) / (M) +
    np.abs(red - M) / (M)
)

hot = (
    blue -
    0.45 * red -
    0.08
)

In [ ]:
blue

In [ ]:
stack = np.stack([
    blue,
    green,
    red,
    nir,
    ndvi,
    wi,
    hot
], axis=-1)

h, w, n = stack.shape

x = stack.reshape(-1, n)

In [ ]:
h*w

In [ ]:
w

In [ ]:
x[23894355]

In [ ]:
mask = (
    np.all(np.isfinite(x), axis=1) &
    np.all(x[:, 0:4] != -9999, axis=1)
)

result = np.full(x.shape[0], 255, dtype=np.uint8)

result[mask] = dt.predict(x[mask])

classification = result.reshape(h, w)

In [ ]:
result

In [ ]:
classification

In [ ]:
plt.imshow(classification)

plt.show()

In [ ]:
classification

In [ ]:
np.unique(classification)

In [ ]:
img = classification.astype(float)

img[img == 255] = np.nan

plt.imshow(img)

plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches


img = classification.astype(float)
img[img == 255] = np.nan

cores = [
    "limegreen",   # clear
    "white",       # cloud
    "black"        # shadow
]

cmap = ListedColormap(cores)

plt.figure(figsize=(10, 10))

plt.imshow(
    img,
    cmap=cmap,
    interpolation="nearest",
    vmin=0,
    vmax=len(le.classes_) - 1
)

plt.axis("off")
plt.title("Decision Tree Classification")

patches = [
    mpatches.Patch(
        color=cores[i],
        label=le.classes_[i]
    )
    for i in range(len(le.classes_))
]

plt.legend(
    handles=patches,
    loc="lower left"
)

plt.show()

In [ ]:
with rasterio.open(bands["blue"]) as src:
    profile = src.profile.copy()

profile.update(
    dtype=rasterio.uint8,
    count=1,
    compress="lzw",
    nodata=255
)

with rasterio.open("classificacao_dt.tif", "w", **profile) as dst:
    dst.write(classification.astype(rasterio.uint8), 1)

print("Arquivo salvo: classificacao_dt.tif")